In [1]:
from collections import defaultdict
from collections import deque
import timeit

In [2]:
input_filename = "day4_input.txt"

In [3]:
def part1_v1(input_filename):

    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    pos = {}
    xmax = len(diagram[0])
    ymax = len(diagram)
    for i in range(xmax):
        for j in range(ymax):
            pos[(i,j)] = diagram[ymax-1-j][i]

    for i in range(-1, xmax+1):
        pos[(i, ymax)] = '.'
        pos[(i, -1)] = '.'

    for j in range(ymax):
        pos[(xmax, j)] = '.'
        pos[(-1, j)] = '.'

    def count_paper(x, y, pos):
        paper = 0
        for i in [x-1, x, x+1]:
            for j in [y-1, y, y+1]:
                if (i!=x) or (j!=y):
                    if pos[(i,j)] == '@':
                        paper+=1
        return paper
    

    accessible_roles = 0

    test_pos = set()
    for i in range(xmax):
        for j in range(ymax):
            if pos[(i,j)] == '@':
                if count_paper(i, j, pos) < 4:
                    accessible_roles+=1

                    test_pos.add((i,j))

    return accessible_roles

part1_v1(input_filename)

1370

In [4]:
def part1_v2(input_filename):

    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    count = 0

    #Parsing and base case in one
    xmax = len(diagram)
    ymax = len(diagram[0])
    for i in range(xmax):
        for j in range(ymax):
            paper = (i, j)

            if diagram[i][j] == '@':
                paper_count = 0

                for (dx, dy) in directions:

                    prop = (i+dx, j+dy)
                    if (0<=prop[0]) and (prop[0]<xmax) and (0<=prop[1]) and (prop[1]<ymax):
                        if diagram[prop[0]][prop[1]] == '@':
                            paper_count += 1
                    
                if paper_count < 4:
                    count+=1
    return count

part1_v2(input_filename)

1370

In [5]:
iters = 10*4
execution_time = timeit.timeit(lambda: part1_v1(input_filename), number=iters)
print(f"Average Execution time version 1: {execution_time/iters} seconds")
execution_time = timeit.timeit(lambda: part1_v2(input_filename), number=iters)
print(f"Average Execution time version 2: {execution_time/iters} seconds")

Average Execution time version 1: 0.00829023229998711 seconds
Average Execution time version 2: 0.008127079175028484 seconds


# part 2

In [6]:
def part2_v1(input_filename):
    #OG solution
    class PaperMess:
        def __init__(self, diagram):

            self.directions = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]
            
            self.pos = {}
            xmax = len(diagram[0])
            ymax = len(diagram)
            for i in range(xmax):
                for j in range(ymax):
                    self.pos[(i,j)] = diagram[ymax-1-j][i]

            for i in range(-1, xmax+1):
                self.pos[(i, ymax)] = '.'
                self.pos[(i, -1)] = '.'

            for j in range(ymax):
                self.pos[(xmax, j)] = '.'
                self.pos[(-1, j)] = '.'

            self.adj = defaultdict(set)
            self.paper_counts = defaultdict(int)
            for i in range(xmax):
                for j in range(ymax):
                    if self.pos[(i, j)] == '@':
                        self.paper_counts[(i,j)] = 0
                        for (dx, dy) in self.directions:
                            if self.pos[(i+dx, j+dy)] == '@':
                                self.paper_counts[(i, j)]+=1
                                self.adj[(i,j)].add((i+dx, j+dy))

            self.clearable = deque()
            self.queued_to_clear = set()
        
        def clear(self):
            #initial search
            for paper, count in self.paper_counts.items():
                if count < 4:
                    self.clearable.append(paper)
                    self.queued_to_clear.add(paper)

            #main search
            while self.clearable:
                paper = self.clearable.popleft()

                #clear it
                self.pos[paper] = '.'

                #update nearby
                for nearby_paper in self.adj[paper]:
                    self.paper_counts[nearby_paper] += -1

                    if (self.paper_counts[nearby_paper]<4) and (nearby_paper not in self.queued_to_clear):
                        self.clearable.append(nearby_paper)
                        self.queued_to_clear.add(nearby_paper)
    

    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    mess = PaperMess(diagram)
    mess.clear()

    return len(mess.queued_to_clear)


In [7]:
# Faster Solution
def part2_v2(input_filename):
    
    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    adj = defaultdict(set)
    counts = defaultdict(int)
    clearable = set()
    clear_queue = deque()

    #Parsing and base case in one
    xmax = len(diagram)
    ymax = len(diagram[0])
    for i in range(xmax):
        for j in range(ymax):
            paper = (i, j)

            if diagram[i][j] == '@':
                counts[paper] = 0

                for (dx, dy) in directions:

                    prop = (i+dx, j+dy)
                    if (0<=prop[0]) and (prop[0]<xmax) and (0<=prop[1]) and (prop[1]<ymax):
                        if diagram[prop[0]][prop[1]] == '@':
                            adj[paper].add(prop)
                            counts[paper] += 1
                    
                if counts[paper] < 4:
                    clear_queue.append(paper)
                    clearable.add(paper)

    # Main search
    while clear_queue:
        paper = clear_queue.popleft()

        #clear by updating nearby
        for nearby_paper in adj[paper]:
            counts[nearby_paper] += -1

            if (counts[nearby_paper]<4) and (nearby_paper not in clearable):
                clear_queue.append(nearby_paper)
                clearable.add(nearby_paper)

    return len(clearable)



In [8]:
iters = 10*4
execution_time = timeit.timeit(lambda: part2_v1(input_filename), number=iters)
print(f"Average Execution time version 1: {execution_time/iters} seconds")
execution_time = timeit.timeit(lambda: part2_v2(input_filename), number=iters)
print(f"Average Execution time version 2: {execution_time/iters} seconds")

Average Execution time version 1: 0.021601492700028757 seconds
Average Execution time version 2: 0.019994837500053108 seconds
